# Week 08 — Home exercise 2: Summarizing the panel

**Solution proposal.**

What the panel can answer, a summary that does not add up, the baseline pattern that turns a level
into an anomaly, and homemade income bands checked against the World Bank's own.

In [1]:
import numpy as np
import pandas as pd


def load_emissions(path):
    """
    Read the emissions file and return it ready to use.

    Parameters
    ----------
    path : str
        Path to the emissions CSV file.

    Returns
    -------
    DataFrame
        One row per country per year, with rows missing total emissions
        dropped and emissions per person added as a column.
    """
    emissions = pd.read_csv(path)

    emissions = emissions.dropna(subset=["co2_total"])
    emissions["co2_pc"] = emissions["co2_total"] * 1_000_000 / emissions["population"]

    return emissions


co2 = load_emissions("../data/co2_emissions.csv")

print(co2.shape)

(5904, 12)


## 1. What can you actually study?

`.count()` with no column selected counts the non-missing values of **every** column at once, one
row per year.

In [2]:
coverage = co2.groupby("year").count()

coverage.loc[[2000, 2010, 2020, 2021, 2022, 2023]]

,country,code,co2_total,population,urban,gdp_pc,electricity,agriculture,nat_resources,renew_energy,co2_pc
year,,,,,,,,,,,
2000,246,246,246,246,246,237,242,243,233,245,246
2010,246,246,246,246,246,243,245,243,240,246,246
2020,246,246,246,246,246,241,245,244,238,246,246
2021,246,246,246,246,246,240,245,244,233,203,246
2022,246,246,246,246,246,240,245,244,0,66,246
2023,246,246,246,246,246,235,245,244,0,0,246


Every entity has all 24 years of `co2_total`, `population` and `urban`, and the other indicators
have small gaps throughout — except two, which have a cliff at the end:

- `nat_resources` stops after **2021**, with nothing at all for 2022 or 2023.
- `renew_energy` thins out in 2021 and stops after **2022**.

So a study of 2023 can use emissions, population, urbanization, GDP, electricity access and
agricultural land, and cannot use natural-resource rents or renewable energy at all. Anyone who wants
those two has to choose an older year, and should say that they did.

This is the same publication lag the lecture found with `.size()` against `.count()`, seen across
every column at once. One line, and it tells you which questions are open before you ask any of them.

In [3]:
gdp_years = co2.groupby("country")["gdp_pc"].count()

print("entities:                 ", len(gdp_years))
print("complete 24-year series:  ", (gdp_years == 24).sum())
print("no gdp_pc at all:         ", (gdp_years == 0).sum())
print()
print(gdp_years[gdp_years < 24].sort_values())

entities:                  246
complete 24-year series:   233
no gdp_pc at all:          3

country
British Virgin Islands        0
Gibraltar                     0
Korea, Dem. People's Rep.     0
Eritrea                      12
Cayman Islands               18
Yemen, Rep.                  19
Cuba                         21
American Samoa               21
Guam                         21
Northern Mariana Islands     21
Virgin Islands (U.S.)        21
Syrian Arab Republic         23
Turks and Caicos Islands     23
Name: gdp_pc, dtype: int64


Grouping by entity instead of by year turns the same question round. **233 of the 246 entities have
the complete GDP series, and three have none at all** — British Virgin Islands, Gibraltar and North
Korea. The ten in between are mostly small territories and countries in conflict, where statistics
stop being collected: Eritrea, Yemen, Syria.

That pattern is worth noticing, because it is not random. A study of GDP and emissions that drops
the incomplete series is not dropping thirteen random rows; it is dropping the war zones and the
territories, and its conclusions are about the rest.

## 2. Income groups, and a total that does not add up

The `.loc` line after `np.select` is the one that matters: without it, every row with no `gdp_pc`
fails all three conditions, falls through to `default`, and is labeled high income.

In [4]:
conditions = [
    co2["gdp_pc"] < 1_000,
    co2["gdp_pc"] < 10_000,
    co2["gdp_pc"] < 50_000,
]

co2["income_group"] = np.select(conditions, ["low", "lower-middle", "upper-middle"], default="high")
co2.loc[co2["gdp_pc"].isna(), "income_group"] = None

year_2023 = co2[co2["year"] == 2023]

by_income = year_2023.groupby("income_group").agg(
    entities=("co2_total", "count"),
    total_co2=("co2_total", "sum"),
    mean_pc=("co2_pc", "mean"),
)

by_income.round(2)

,entities,total_co2,mean_pc
income_group,,,
high,26,22048.12,8.96
low,17,78.02,0.19
lower-middle,116,141982.29,2.06
upper-middle,76,155918.02,7.68


In [5]:
print("sum of the groups:", by_income["total_co2"].sum().round(1))
print("sum of the column:", year_2023["co2_total"].sum().round(1))
print("difference:       ", (year_2023["co2_total"].sum() - by_income["total_co2"].sum()).round(1))
print()
print("rows with no income_group:", year_2023["income_group"].isna().sum())
print(year_2023[year_2023["income_group"].isna()]["country"].tolist())

sum of the groups: 320026.5
sum of the column: 320155.6
difference:        129.2

rows with no income_group: 11
['American Samoa', 'British Virgin Islands', 'Cuba', 'Eritrea', 'Gibraltar', 'Guam', "Korea, Dem. People's Rep.", 'Northern Mariana Islands', 'Syrian Arab Republic', 'Virgin Islands (U.S.)', 'Yemen, Rep.']


**No, they do not, and 129.2 million tonnes are missing.**

Eleven entities have no `gdp_pc` in 2023, so they have no `income_group`, so `groupby` leaves them
out. The four group totals are each correct, and their sum is not the total. The eleven are the same
kind of entity as in question 1 — territories, and countries whose statistics have stopped.

The check that catches it is one line, and it is the line to write after every `groupby` on a key
that could be missing. (The total here is also inflated many times over by the aggregates, which this
table still contains. That is a separate problem, and the last part of the exercise deals with it.)

## A baseline over part of the period

The baseline is an average over **2000–2009 only**, and it has to end up beside every row from 2000
to 2023. `transform("mean")` on its own computes over every row in the group — all 24 years — so it
gives the wrong average.

So the baseline becomes its own table, and comes back with a merge.

In [6]:
baseline_years = (co2["year"] >= 2000) & (co2["year"] <= 2009)

baseline = (
    co2[baseline_years]
    .groupby("country")["co2_pc"]
    .mean()
    .reset_index()
    .rename(columns={"co2_pc": "baseline_pc"})
)

print(baseline.shape)
baseline.head(3).round(3)

(246, 2)


,country,baseline_pc
0,Afghanistan,0.078
1,Africa Eastern and Southern,1.102
2,Africa Western and Central,0.489


`reset_index()` turns the grouped Series into a two-column table, and `rename` stops the merge from
producing `co2_pc_x` and `co2_pc_y`. One row per entity, so the merge is many-to-one — and we say so.

In [7]:
before = len(co2)

co2 = co2.merge(baseline, on="country", how="left", validate="many_to_one")
co2["anomaly"] = co2["co2_pc"] - co2["baseline_pc"]

print("rows before:", before)
print("rows after: ", len(co2))

rows before: 5904
rows after:  5904


In [8]:
top_2023 = (
    co2[co2["year"] == 2023]
    .sort_values("anomaly", ascending=False)
    [["country", "population", "co2_pc", "baseline_pc", "anomaly"]]
)

top_2023.head(5).round(2)

,country,population,co2_pc,baseline_pc,anomaly
3911,New Caledonia,289870,16.78,10.59,6.20
1079,China,1410710000,9.23,4.49,4.74
4151,Oman,5049269,18.96,14.62,4.34
2135,Greenland,56865,10.29,6.43,3.85
1487,East Asia & Pacific (IDA & IBRD countries),2111139152,7.11,3.63,3.48


### The one that is not a country

**`East Asia & Pacific (IDA & IBRD countries)`** is a World Bank grouping, not a country, and it is
sitting fifth in a list of countries.

Three things give it away, in increasing order of reliability:

1. **The name.** A region and a parenthesis full of acronyms is not a country. This works here and
   will not work in general — plenty of real countries have long names, and plenty of aggregates have
   short ones.
2. **The population.** 2.1 billion people is more than any country on Earth has.
3. **A second table that says so.** `country_info.csv` labels every entity with a region, and marks
   the groupings `"Aggregates"`. That is the only method that scales, and it is the one the next part
   uses.

### There is a way to do it with `transform`, too

`transform` averages over whatever values the group holds. So blank out the values outside the
baseline period first, and its average is the baseline. Both routes give identical numbers.

In [9]:
outside_baseline = (co2["year"] < 2000) | (co2["year"] > 2009)

co2["pc_in_baseline"] = co2["co2_pc"]
co2.loc[outside_baseline, "pc_in_baseline"] = np.nan

co2["baseline_by_transform"] = co2.groupby("country")["pc_in_baseline"].transform("mean")

print("largest difference between the two baselines:",
      (co2["baseline_by_transform"] - co2["baseline_pc"]).abs().max())

largest difference between the two baselines: 0.0


Which to prefer is a matter of what you want to keep. The merge route leaves you with the baseline
as a small table of its own — one row per entity, easy to inspect, save or reuse. The `transform`
route is shorter, and depends on a helper column whose whole purpose is to be partly empty, which is
the kind of trick a reader has to stop and decode.

## Finally: our bands against the World Bank's

In [10]:
info = pd.read_csv("../data/country_info.csv")

panel = co2.merge(info[["code", "region", "incomeLevel"]], on="code", how="left", validate="many_to_one")
countries_2023 = panel[(panel["region"] != "Aggregates") & (panel["year"] == 2023)].copy()

print("countries in 2023:", len(countries_2023))
print(countries_2023["incomeLevel"].value_counts(dropna=False))

countries in 2023: 203
incomeLevel
High income            77
Upper middle income    56
Lower middle income    46
Low income             24
Name: count, dtype: int64


### 1. How often do they agree?

Grouping on **two** keys counts every combination that occurs.

In [11]:
countries_2023.groupby(["incomeLevel", "income_group"]).size()

incomeLevel          income_group
High income          high            24
                     upper-middle    47
Low income           low             16
                     lower-middle     4
Lower middle income  low              1
                     lower-middle    45
Upper middle income  lower-middle    42
                     upper-middle    13
dtype: int64

Agreement is on the diagonal: 24 + 16 + 45 + 13 = **98 of the 192 countries** that have a GDP figure.
Barely half.

And the disagreements all go the same way. 47 of the World Bank's high-income countries are
"upper-middle" by our bands, and 42 of its upper-middle countries are "lower-middle". Our bands are not
noisy; they are **too high**, consistently, at both of the upper boundaries.

### 2. With the bank's own thresholds

The World Bank classifies countries by **GNI per person**, converted to dollars with its own "Atlas"
method, and revises the thresholds every July. The classification for 2025, based on 2023 income,
used these thresholds: low up to 1 145 dollars, lower-middle up to 4 515, upper-middle up to 14 005,
and high above that. (The thresholds you find may be a newer set, a little different.)

To compare row by row, both classifications need the same labels, so the bands are built with the
bank's names.

In [12]:
labels = ["Low income", "Lower middle income", "Upper middle income"]


def income_level_from(gdp, cuts):
    """Label GDP per person with the World Bank's income levels, using the three cut-offs given."""
    conditions = [gdp < cuts[0], gdp < cuts[1], gdp < cuts[2]]
    return np.select(conditions, labels, default="High income")


countries_2023["band_ours"] = income_level_from(countries_2023["gdp_pc"], (1_000, 10_000, 50_000))
countries_2023["band_bank"] = income_level_from(countries_2023["gdp_pc"], (1_145, 4_515, 14_005))
countries_2023.loc[countries_2023["gdp_pc"].isna(), ["band_ours", "band_bank"]] = None

print("countries with a band:     ", countries_2023["band_ours"].notna().sum())
print("agree, our thresholds:     ", (countries_2023["band_ours"] == countries_2023["incomeLevel"]).sum())
print("agree, the bank's:         ", (countries_2023["band_bank"] == countries_2023["incomeLevel"]).sum())

countries with a band:      192
agree, our thresholds:      98
agree, the bank's:          184


**From 98 to 184 of 192.** Almost all of the disagreement was the thresholds.

In [13]:
has_band = countries_2023["band_bank"].notna()
still_wrong = countries_2023[has_band & (countries_2023["band_bank"] != countries_2023["incomeLevel"])]

still_wrong[["country", "gdp_pc", "band_bank", "incomeLevel"]].sort_values("gdp_pc").round(0)

,country,gdp_pc,band_bank,incomeLevel
3167,Lesotho,921.0,Low income,Lower middle income
4367,Philippines,3804.0,Lower middle income,Upper middle income
4991,Sri Lanka,3815.0,Lower middle income,Upper middle income
3575,"Micronesia, Fed. Sts.",3933.0,Lower middle income,Upper middle income
5783,Viet Nam,4323.0,Lower middle income,Upper middle income
3839,Naoero,13603.0,Upper middle income,High income
4559,Russian Federation,13987.0,Upper middle income,High income
239,Argentina,14262.0,High income,Upper middle income


### Why the first bands disagreed, and why the corrected ones still do

**The first bands were simply set too high.** 50 000 and 10 000 are round numbers, not the bank's
numbers, and a classification is only as good as its cut-offs. That is the general lesson: labels
you build yourself from a continuous variable carry every assumption in the thresholds, and nothing
in the output reveals it.

**The remaining eight cannot be fixed by thresholds**, because we are measuring a different thing:

- **GDP is not GNI.** GDP counts what is produced inside a country; GNI adds the income its residents
  earn from abroad and subtracts what foreign owners take out. Where many residents work abroad — the
  Philippines is the textbook case — GNI per person can sit well above GDP per person.
- **The conversion differs.** Our `gdp_pc` is in current dollars at that year's exchange rate; the
  Atlas method smooths exchange rates over three years.
- **The year may differ.** The classification in `country_info.csv` is whatever the bank published
  when the file was downloaded, which may be based on a later year's income than 2023.
- **Several sit right at a boundary.** Russia at 13 987 against a threshold of 14 005, and Argentina
  at 14 262, are within two percent of it. At that distance any difference in method or year decides
  the label.

### Things worth noticing

- **`.count()` with no column is the cheapest data audit available.** One line per question you could
  ask, before you ask it.
- **`transform` and `merge` solve the same problem in different circumstances.** `transform` fits when
  the statistic is computed over the same rows it has to sit beside; a small table plus a merge fits
  when it is computed over a subset, a reference year, or a different dataset.
- **Grouping on two keys is a cross-tabulation.** Every combination that occurs gets a count, and the
  pattern of the off-diagonal cells is often more informative than the agreement itself.

### What this notebook does NOT do

- The baseline is an unweighted mean of whatever years happen to be present between 2000 and 2009. An
  entity observed in only two of those years gets a baseline anyway, and a much noisier one — nothing
  here reports how many years each baseline was built from, and it should.
- The 2023 summary in question 2 still contains the aggregates, so its totals and means are not about
  countries. Only the last part removes them.
- The comparison with the World Bank uses one year. A country close to a threshold can move between
  levels from one year to the next, so agreement in 2023 says nothing about any other year.